# Mappa "Sardegna: Gemme Nascoste"

Questo notebook genera una mappa interattiva Leaflet analoga a `Sardegna_Gemme_Nascoste.html`
(interrogabile per mese/anno, con classifica delle "gemme" del mese), ma usando i **dati reali del
progetto** invece dei dati dimostrativi: l'indicatore combinato prodotto in
`data/indicatore_gem_score/04_gemme_nascoste.ipynb`, che unisce:

- **Indice di attrattività** — `data/indicatore_attrattivita/output_attrattivita`
- **Indice di overtourism** — `data/indice_overtourism`

### Come vengono identificate le "gemme"

Il file sorgente `indicatore_gem_score.csv` classifica ogni comune, mese per mese, in 4 **quadranti**
confrontando indice di attrattività e indice di overtourism con le rispettive mediane del pannello:

| Quadrante | Attrattività | Overtourism | Significato |
|-----------|-------------|-------------|-------------|
| **Q1 – Gemma Nascosta** 🌟 | ≥ mediana | < mediana | quello che stiamo cercando |
| Q2 – Destinazione Popolare 🏖️ | ≥ mediana | ≥ mediana | bella ma affollata |
| Q3 – Territorio Autentico 🌄 | < mediana | < mediana | potenziale inespresso |
| Q4 – Zona Satura ⚠️ | < mediana | ≥ mediana | poco attraente e affollata |

La mappa mostra tutti i comuni colorati per quadrante e, nel pannello laterale, la classifica dei
comuni **Q1** del mese selezionato, ordinati per `gem_score_normalized`.

> **Nota metodologica:** nel notebook `04_gemme_nascoste.ipynb` il gem score è calcolato come
> `indice_attrattivita * (1 - indice_overtourism / 100)`. Nei CSV attuali `indice_overtourism` è già
> in scala 0–1 (non 0–100), quindi quella divisione per 100 lo rende quasi ininfluente sul punteggio
> globale: `gem_score_normalized` finisce per riflettere quasi solo l'attrattività. Non tocchiamo qui
> il file/notebook originale (non è stato chiesto), ma **l'eleggibilità a "gemma" usata in questa
> mappa si basa sul quadrante Q1** (calcolato confrontando le mediane grezze di attrattività e
> overtourism, non affetto da questo problema di scala), non sul solo gem_score. All'interno del
> sottoinsieme Q1 il gem_score resta comunque un ordinamento ragionevole, perché lì l'overtourism è
> già per definizione sotto la mediana.

## 1 · Configurazione

Percorsi relativi alla cartella del notebook (`data/prova/`), coerenti con gli altri notebook del progetto.

In [14]:
# Installazione minima: il notebook resta eseguibile anche in un ambiente nuovo.
import subprocess, sys
for pkg in ['pandas']:
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

import json, math, os
from pathlib import Path
import pandas as pd

ROOT = Path('../')
DATA = ROOT / 'data'
OUT = ROOT / 'data' / 'mappa_gemme_nascoste'

GEM_CSV = DATA / 'indicatore_gem_score' / 'output_gem_score' / 'indicatore_gem_score.csv'
MUNI_GEOJSON = OUT / 'sardinia_municipalities.geojson'
REGION_GEOJSON = OUT / 'sardinia_region.geojson'
OUT_HTML = OUT / 'mappa_gemme_nascoste.html'

for f in [GEM_CSV, MUNI_GEOJSON, REGION_GEOJSON]:
    if not f.exists():
        raise FileNotFoundError(f"File non trovato: {f}")

print("Configurazione OK")
print(f"  GEM_CSV        : {GEM_CSV}")
print(f"  MUNI_GEOJSON   : {MUNI_GEOJSON}")
print(f"  REGION_GEOJSON : {REGION_GEOJSON}")
print(f"  OUT_HTML       : {OUT_HTML.resolve()}")

Configurazione OK
  GEM_CSV        : ../data/indicatore_gem_score/output_gem_score/indicatore_gem_score.csv
  MUNI_GEOJSON   : ../data/mappa_gemme_nascoste/sardinia_municipalities.geojson
  REGION_GEOJSON : ../data/mappa_gemme_nascoste/sardinia_region.geojson
  OUT_HTML       : /home/davide/codice/progetto_git/sardegna-overtourism-aida26/data/mappa_gemme_nascoste/mappa_gemme_nascoste.html


## 2 · Centroidi comunali

Il CSV dell'indicatore non contiene lat/lng. Li calcoliamo dal poligono di ogni comune in
`sardinia_municipalities.geojson` (stesso schema di `comuni_sardegna.geojson`), usando il centroide
geometrico (formula dell'area con segno) del poligono più grande in caso di `MultiPolygon`, gestendo
correttamente i buchi (enclave).

In [15]:
def _ring_area_centroid(ring):
    """Area con segno e centroide di un singolo anello [[lon,lat], ...]."""
    a = cx = cy = 0.0
    for i in range(len(ring) - 1):
        x0, y0 = ring[i]
        x1, y1 = ring[i + 1]
        cross = x0 * y1 - x1 * y0
        a += cross
        cx += (x0 + x1) * cross
        cy += (y0 + y1) * cross
    a *= 0.5
    if a == 0:
        return 0.0, 0.0, 0.0
    return a, cx / (6 * a), cy / (6 * a)


def _polygon_centroid(rings):
    """Centroide di un poligono con eventuali buchi: rings = [esterno, buco1, ...]."""
    a_tot = cx_tot = cy_tot = 0.0
    for ring in rings:
        a, cx, cy = _ring_area_centroid(ring)
        a_tot += a
        cx_tot += cx * a
        cy_tot += cy * a
    if a_tot == 0:
        return None
    return cx_tot / a_tot, cy_tot / a_tot


def geometry_centroid(geometry):
    """Centroide del poligono più esteso di una geometry Polygon/MultiPolygon GeoJSON.
    Ritorna (lat, lng) oppure None."""
    gtype = geometry["type"]
    if gtype == "Polygon":
        polygons = [geometry["coordinates"]]
    elif gtype == "MultiPolygon":
        polygons = geometry["coordinates"]
    else:
        return None

    best, best_area = None, -1
    for rings in polygons:
        a, _, _ = _ring_area_centroid(rings[0])
        if abs(a) > best_area:
            best_area, best = abs(a), rings

    c = _polygon_centroid(best)
    if c is None:
        return None
    lng, lat = c
    return lat, lng


print("Funzioni di centroide definite.")

Funzioni di centroide definite.


In [16]:
with open(MUNI_GEOJSON, encoding='utf-8') as f:
    muni_geo = json.load(f)

geo_by_code = {}
for feat in muni_geo['features']:
    props = feat['properties']
    code_istat = int(props['com_istat_code_num'])
    centroid = geometry_centroid(feat['geometry'])
    geo_by_code[code_istat] = {
        'lat': centroid[0] if centroid else None,
        'lng': centroid[1] if centroid else None,
        'province': props.get('prov_name'),
    }

with open(REGION_GEOJSON, encoding='utf-8') as f:
    region_geo = json.load(f)
region_geometry = region_geo['features'][0]['geometry']

print(f"Geometrie comunali caricate : {len(geo_by_code)}")
print(f"Geometria regione           : {region_geometry['type']}")

Geometrie comunali caricate : 377
Geometria regione           : MultiPolygon


## 3 · Caricamento dell'indicatore gem score

Carichiamo `indicatore_gem_score.csv` (output di `04_gemme_nascoste.ipynb`) e costruiamo l'elenco
ordinato dei 48 step mensili (2022-01 → 2025-12) presenti nel pannello.

In [17]:
df = pd.read_csv(GEM_CSV)
df = df.drop(columns=[c for c in df.columns if c.startswith('Unnamed')])
df['codice_istat'] = df['codice_istat'].astype(int)

steps = sorted(df[['anno', 'mese']].drop_duplicates().itertuples(index=False, name=None))
step_index = {s: i for i, s in enumerate(steps)}
n_steps = len(steps)

print(f"Righe totali        : {len(df):,}")
print(f"Comuni              : {df['comune'].nunique()}")
print(f"Step mensili        : {n_steps}  ({steps[0]} → {steps[-1]})")
print(f"Distribuzione quadranti (tutte le righe):")
print(df['quadrante'].value_counts().to_string())

Righe totali        : 17,808
Comuni              : 371
Step mensili        : 48  ((2022, 1) → (2025, 12))
Distribuzione quadranti (tutte le righe):
quadrante
Q3    7374
Q2    6194
Q4    2766
Q1    1474


## 4 · Struttura compatta per la mappa

Per ogni comune costruiamo un oggetto con i campi fissi (nome, provincia, area, coordinate,
attrattività — costante nel tempo nell'attuale pipeline) e gli array mensili (48 valori, uno per
step) di gem score, overtourism, quadrante e dei 4 indicatori grezzi usati nei popup.

In [18]:
QUAD_CODE = {'Q1': '1', 'Q2': '2', 'Q3': '3', 'Q4': '4'}


def r(v, nd):
    if v is None or (isinstance(v, float) and math.isnan(v)):
        return None
    return round(float(v), nd)


comuni = []
missing_coords = []
for cod, g in df.groupby('codice_istat', sort=False):
    g = g.set_index(['anno', 'mese'])
    name = g['comune'].iloc[0]

    gem = [None] * n_steps
    ovt = [None] * n_steps
    quad = [None] * n_steps
    dt = [None] * n_steps
    it = [None] * n_steps
    dr = [None] * n_steps
    ul = [None] * n_steps
    nz = [None] * n_steps

    for (anno, mese), row in g.iterrows():
        idx = step_index[(anno, mese)]
        gem[idx] = r(row['gem_score_normalized'], 1)
        ovt[idx] = r(row['indice_overtourism'] * 100, 2)
        quad[idx] = QUAD_CODE.get(row['quadrante'], '4')
        dt[idx] = r(row['densita_turistica'], 1)
        it[idx] = r(row['intensita_turistica_presenze'], 3)
        dr[idx] = r(row['densita_ricettiva'], 2)
        ul[idx] = r(row['utilizzazione_lorda_pct'], 1)
        nz[idx] = [
            r(row['densita_turistica_norm'], 2),
            r(row['intensita_turistica_norm'], 2),
            r(row['densita_ricettiva_norm'], 2),
            r(row['utilizzazione_lorda_norm'], 2),
        ]

    geo = geo_by_code.get(cod, {})
    lat, lng = geo.get('lat'), geo.get('lng')
    if lat is None:
        missing_coords.append(name)
        continue

    comuni.append({
        'n': name, 'i': cod, 'pv': geo.get('province'),
        'area': r(g['area_km2'].iloc[0], 2),
        'lat': r(lat, 5), 'lng': r(lng, 5),
        'attr': r(g['indice_attrattivita'].iloc[0] * 100, 1),
        'gem': gem, 'ovt': ovt, 'quad': quad,
        'dt': dt, 'it': it, 'dr': dr, 'ul': ul, 'nz': nz,
    })

print(f"Comuni con marker posizionato : {len(comuni)}")
if missing_coords:
    print(f"Comuni senza geometria (esclusi): {missing_coords}")

Comuni con marker posizionato : 371


## 5 · Meta-dati e assemblaggio

Etichette, icone e colori dei quadranti (condivisi con la mappa Leaflet), più l'elenco degli step per
popolare le tendine mese/anno lato client.

In [19]:
MONTH_NAMES = ['Gennaio', 'Febbraio', 'Marzo', 'Aprile', 'Maggio', 'Giugno',
               'Luglio', 'Agosto', 'Settembre', 'Ottobre', 'Novembre', 'Dicembre']

meta = {
    'title': 'Sardegna: Gemme Nascoste (prova)',
    'generated': pd.Timestamp.now().strftime('%Y-%m-%d'),
    'data_note': ("Indicatore gem score comunale calcolato in "
                  "data/indicatore_gem_score/04_gemme_nascoste.ipynb, a partire da "
                  "indice di attrattività (OSM) e indice di overtourism ISTAT."),
    'years': sorted({s[0] for s in steps}),
    'months_names': MONTH_NAMES,
    'steps': [list(s) for s in steps],
    'n_comuni': len(comuni),
    'quad_labels': {'1': 'Gemma Nascosta', '2': 'Destinazione Popolare',
                     '3': 'Territorio Autentico', '4': 'Zona Satura'},
    'quad_desc': {
        '1': 'alta attrattività, bassa pressione turistica',
        '2': 'alta attrattività, alta pressione turistica',
        '3': 'bassa attrattività, bassa pressione turistica',
        '4': 'bassa attrattività, alta pressione turistica',
    },
    'quad_icons': {'1': '\U0001F31F', '2': '\U0001F3D6\uFE0F', '3': '\U0001F304', '4': '\u26A0\uFE0F'},
    'quad_colors': {'1': '#1a9850', '2': '#e67e22', '3': '#3498db', '4': '#c0392b'},
}

embedded_data = {'meta': meta, 'region': region_geometry, 'comuni': comuni}
data_json = json.dumps(embedded_data, ensure_ascii=False, separators=(',', ':'))
print(f"Dimensione JSON embedded-data: {len(data_json)/1024:.1f} KB")

Dimensione JSON embedded-data: 1271.1 KB


## 6 · Anteprima: le gemme dell'ultimo mese disponibile

Controllo rapido prima di generare l'HTML: quanti comuni Q1 nell'ultimo mese del pannello e quali
sono i primi 10 per `gem_score_normalized`.

In [20]:
ultimo_anno, ultimo_mese = steps[-1]
preview = df[(df.anno == ultimo_anno) & (df.mese == ultimo_mese) & (df.quadrante == 'Q1')]
preview = preview.sort_values('gem_score_normalized', ascending=False)

print(f"Mese di riferimento: {ultimo_mese:02d}/{ultimo_anno}  →  {len(preview)} comuni Q1 (gemme eleggibili)")
display(preview[['comune', 'indice_attrattivita', 'indice_overtourism', 'gem_score_normalized']]
        .head(10).reset_index(drop=True))

Mese di riferimento: 12/2025  →  39 comuni Q1 (gemme eleggibili)


,comune,indice_attrattivita,indice_overtourism,gem_score_normalized
0,Burcei,0.653853,0.0000,75.372115
1,Cardedu,0.499600,0.0000,57.590828
2,Seui,0.485248,0.0000,55.936352
3,Seulo,0.475286,0.0000,54.788067
4,Ussassai,0.475970,0.0019,54.762565
5,Tuili,0.438209,0.0000,50.513975
6,Desulo,0.428263,0.0019,49.273661
7,Ittiri,0.414578,0.0026,47.665754
8,Sedini,0.410144,0.0000,47.278809
9,Seneghe,0.400746,0.0000,46.195493


## 7 · Generazione della mappa HTML

Stesso impianto della mappa allegata (pannello laterale con selettori mese/anno, classifica delle
gemme, popup per comune, interruttore "solo gemme sulla mappa"), riadattato ai campi realmente
disponibili nella nostra pipeline. La mappa è **auto-contenuta**: i dati sono incorporati come JSON
nell'HTML, Leaflet è caricato da CDN.

In [21]:
CSS = """
:root{--blue:#1a5276;--sea:#48c9b0;--sand:#f4e4c1;--terr:#cb6015;--olive:#7d8c4e;--ink:#173042;--muted:#587080;--panel:rgba(255,253,248,.96)}
*{box-sizing:border-box}html,body{height:100%;margin:0;font-family:-apple-system,BlinkMacSystemFont,"Segoe UI",sans-serif;color:var(--ink);overflow:hidden}body{background:#dceeea}#map{position:fixed;inset:0}.leaflet-control-zoom a{color:var(--blue)!important}.leaflet-control-attribution{font-size:10px}
#sidebar{position:fixed;z-index:1000;top:14px;right:14px;bottom:14px;width:380px;max-width:calc(100vw - 28px);display:flex;flex-direction:column;background:var(--panel);border:1px solid rgba(26,82,118,.16);box-shadow:0 16px 50px rgba(23,48,66,.23);border-radius:20px;overflow:hidden;backdrop-filter:blur(12px)}
header{padding:24px 24px 16px;background:linear-gradient(135deg,#1a5276,#247d91);color:#fff}header h1{font-size:25px;line-height:1.05;margin:0 0 8px;letter-spacing:-.6px}header p{font-size:12px;line-height:1.45;margin:0;color:#d8f5ef}.eyebrow{font-size:10px;letter-spacing:1.8px;text-transform:uppercase;color:#a8e7d8;font-weight:800;margin-bottom:9px}.disclaimer{margin:12px 24px 0;padding:10px 12px;border-radius:10px;background:#fff5dd;border:1px solid #f0d59c;color:#735221;font-size:10.5px;line-height:1.4}
.controls{padding:14px 24px 4px;display:grid;grid-template-columns:1fr 100px;gap:10px}.controls-single{padding:14px 24px 4px}.field{position:relative}.field label{display:block;color:var(--muted);font-size:10px;font-weight:800;text-transform:uppercase;letter-spacing:.7px;margin-bottom:5px}.field select,.field input{width:100%;border:1px solid #bed4d4;background:#fff;border-radius:8px;padding:8px;color:var(--ink);font-weight:700;font-family:inherit}.field input{font-weight:500}
.searchresults{position:absolute;z-index:1500;top:100%;left:0;right:0;background:#fff;border:1px solid #d7e3e0;border-radius:10px;margin-top:4px;max-height:220px;overflow:auto;box-shadow:0 10px 24px rgba(23,48,66,.18)}.searchresults.empty{display:none}.searchresults div{padding:7px 12px;font-size:12px;cursor:pointer;display:flex;gap:6px;align-items:baseline}.searchresults div:hover{background:#eef6f4}.searchresults .prov{color:var(--muted);font-weight:400;font-size:10.5px}.searchresults .none{padding:8px 12px;color:var(--muted);font-size:11px;cursor:default}
.qfilter{padding:11px 24px 12px;display:flex;flex-wrap:wrap;gap:6px;align-items:center;border-bottom:1px solid #e5eeeb}.qchip{--qc:#999;display:flex;align-items:center;gap:5px;padding:5px 10px;border-radius:999px;border:1px solid #d7e3e0;background:#fff;font-size:10.5px;font-weight:700;color:var(--muted);cursor:pointer;user-select:none;transition:.15s}.qchip:hover{border-color:var(--qc)}.qchip.active{background:var(--qc);border-color:var(--qc);color:#fff}.qfilter-all{font-size:10px;color:var(--blue);cursor:pointer;text-decoration:underline;font-weight:700}.qfilter-sep{color:#c3d0ce;font-size:10px}
.list-head{display:flex;justify-content:space-between;align-items:center;padding:15px 24px 9px}.list-head h2{font-size:14px;margin:0}.count{font-size:10px;color:var(--muted)}#rankings{overflow:auto;flex:1;padding:0 13px 10px 24px}.empty{padding:20px;color:#6d7e83;font-size:12px}.card{position:relative;background:#fff;border:1px solid #e1ece9;border-radius:12px;padding:11px 11px 10px 43px;margin:0 10px 8px 0;cursor:pointer;transition:transform .18s,box-shadow .18s,border-color .18s}.card:hover{transform:translateX(-3px);box-shadow:0 5px 16px rgba(26,82,118,.13);border-color:#8fd7ca}.rank{position:absolute;left:11px;top:12px;width:24px;height:24px;border-radius:8px;background:#e8f5f2;color:#147968;font-size:12px;font-weight:900;display:grid;place-items:center}.card:nth-child(-n+3) .rank{background:#f4e4c1;color:#8b5b12}.cardtop{display:flex;justify-content:space-between;gap:8px;align-items:center}.place{font-weight:800;font-size:13px}.score{font-weight:900;font-size:13px;color:#187968}.why{margin:4px 0 8px;font-size:10.5px;line-height:1.35;color:#5e727b}.bars{height:6px;display:flex;gap:2px}.bar{height:100%;border-radius:4px;min-width:4px;background:linear-gradient(90deg,var(--sea),var(--blue))}.meta{font-size:9.5px;color:#829199;margin-top:6px;display:flex;justify-content:space-between}.method{border-top:1px solid #e5eeeb;padding:0 24px}.method summary{padding:13px 0;font-size:12px;font-weight:800;cursor:pointer;color:var(--blue)}.method .inside{font-size:10.5px;line-height:1.45;color:#516872;padding:0 0 15px}.method b{color:var(--blue)}
.legend{position:fixed;left:14px;bottom:14px;z-index:1000;background:var(--panel);border-radius:14px;padding:10px 14px;box-shadow:0 10px 30px rgba(23,48,66,.2);font-size:10.5px;color:var(--ink);backdrop-filter:blur(8px)}.legend div{display:flex;align-items:center;gap:7px;margin:3px 0}.legend i{width:11px;height:11px;border-radius:50%;display:inline-block}
.leaflet-popup-content-wrapper{border-radius:14px}.leaflet-popup-content{margin:15px 17px;min-width:255px}.popup h3{margin:0 0 3px;color:var(--blue);font-size:18px}.popup .sub{font-size:10px;color:var(--muted);margin-bottom:11px}.kv{display:grid;grid-template-columns:1fr 1fr;gap:7px;font-size:10.5px;margin-bottom:8px}.kv span{display:block;color:#78909a;font-size:9px;text-transform:uppercase}.ind{margin:11px 0 8px}.indrow{display:grid;grid-template-columns:130px 1fr 55px;align-items:center;gap:5px;font-size:9.5px;margin:4px 0}.track{height:5px;background:#e6efed;border-radius:5px;overflow:hidden}.fill{height:100%;background:linear-gradient(90deg,var(--sea),var(--blue));border-radius:5px}.bigscore{display:flex;justify-content:space-between;align-items:center;background:#f1f8f5;border-radius:9px;padding:8px 10px;margin-top:10px;font-size:11px}.bigscore strong{font-size:15px;color:#147968}.whyp{font-size:10.5px;line-height:1.35;margin:10px 0 0;color:#587080}.leaflet-tooltip{font-size:11px;border:0;border-radius:7px;box-shadow:0 3px 12px #0002}#loading{position:fixed;inset:0;z-index:2000;display:grid;place-items:center;background:#eaf7f3;color:var(--blue);font-weight:800;letter-spacing:.3px}#loading span{display:inline-block;border:3px solid #bde5dc;border-top-color:var(--blue);width:30px;height:30px;border-radius:50%;animation:spin .8s linear infinite;margin-right:9px;vertical-align:middle}@keyframes spin{to{transform:rotate(360deg)}}
@media(max-width:700px){body{overflow:auto}#sidebar{top:auto;left:8px;right:8px;bottom:8px;width:auto;max-width:none;height:min(67vh,520px);border-radius:17px}header{padding:15px 18px 12px}header h1{font-size:21px}.disclaimer{margin:9px 18px 0;padding:8px}.controls,.controls-single{padding:10px 18px 2px}.qfilter,.list-head,.method{padding-left:18px;padding-right:18px}#rankings{padding-left:18px}.legend{left:8px;bottom:8px}}
"""

BODY_HEAD = """
<div id="loading"><div><span></span>Preparazione della mappa…</div></div>
<div id="map"></div>
<div class="legend" id="legend"></div>
<aside id="sidebar"><header><div class="eyebrow">Esplora · Sardegna</div><h1>Sardegna:<br>Gemme Nascoste</h1><p>Filtra per quadrante o cerca un comune per nome, mese per mese.</p></header>
<div class="disclaimer">Dati reali del progetto: indicatore_gem_score.csv (attrattività OSM + overtourism ISTAT), 371 comuni × 48 mesi (2022-2025).</div>
<div class="controls-single"><div class="field" id="searchwrap"><label for="searchInput">Cerca comune</label><input id="searchInput" type="text" autocomplete="off" placeholder="Nome del comune…"><div id="searchResults" class="searchresults empty"></div></div></div>
<div class="controls"><div class="field"><label for="month">Mese</label><select id="month"></select></div><div class="field"><label for="year">Anno</label><select id="year"></select></div></div>
<div class="qfilter" id="qfilter"></div>
<div class="list-head"><h2 id="listTitle">Comuni</h2><span class="count" id="count">—</span></div><div id="rankings"></div>
<details class="method"><summary>Metodologia</summary><div class="inside">
<p><b>Quadranti:</b> ogni comune/mese è classificato confrontando indice di attrattività e indice di overtourism con le rispettive mediane del pannello. Usa i filtri sopra per scegliere quali quadranti mostrare — sulla mappa e in classifica — oppure lasciali tutti attivi per vedere l'intera Sardegna.</p>
<p><b>Gem score</b> = attrattività × (1 − overtourism), normalizzato 0-100 sull'intero pannello (calcolato in 04_gemme_nascoste.ipynb). Ordina la classifica indipendentemente dal quadrante.</p>
<p><b>Cerca comune:</b> digita un nome per trovarlo ovunque si trovi (anche fuori dai quadranti selezionati) e selezionarlo sulla mappa come con un click.</p>
<p><b>Indicatori mensili:</b> densita' turistica, intensita' turistica (presenze), densita' ricettiva, utilizzazione lorda — dai CSV dell'indice di overtourism.</p>
</div></details></aside>
"""


def build_html(embedded_data_json: str) -> str:
    return f"""<!DOCTYPE html>
<html lang="it">
<head>
<meta charset="utf-8">
<meta name="viewport" content="width=device-width,initial-scale=1">
<title>Sardegna: Gemme Nascoste</title>
<link rel="stylesheet" href="https://unpkg.com/leaflet@1.9.4/dist/leaflet.css" />
<style>{CSS}</style>
</head>
<body>
{BODY_HEAD}
<script type="application/json" id="embedded-data">{embedded_data_json}</script>
<script src="https://unpkg.com/leaflet@1.9.4/dist/leaflet.js"></script>
<script>
"use strict";
const DATA = JSON.parse(document.getElementById('embedded-data').textContent);
const MONTHS = DATA.meta.months_names, YEARS = DATA.meta.years, STEPS = DATA.meta.steps;
const QUAD_KEYS = ['1', '2', '3', '4'];
const STEP_INDEX = {{}};
STEPS.forEach((s, i) => STEP_INDEX[s[0] + '-' + s[1]] = i);
const $ = id => document.getElementById(id);
let month = Math.min(12, Math.max(1, new Date().getMonth() + 1));
let year = YEARS[YEARS.length - 1];
let selectedQuads = new Set(QUAD_KEYS);
let map, markers = new Map(), regionLayer, currentByCode = new Map();

function fmt(v, d = 1) {{ return v == null ? 'n.d.' : Number(v).toLocaleString('it-IT', {{maximumFractionDigits: d}}); }}

function rowFor(c, idx) {{
  if (idx == null) return null;
  const gem = c.gem[idx];
  if (gem == null) return null;
  return {{gem, ovt: c.ovt[idx], quad: c.quad[idx], dt: c.dt[idx], it: c.it[idx], dr: c.dr[idx], ul: c.ul[idx], nz: c.nz[idx] || [0, 0, 0, 0]}};
}}

function markerRadius(gem) {{ return Math.max(5, Math.min(16, 5 + (gem || 0) * 0.11)); }}

function popup(c, r) {{
  const labels = ['Densità turistica', 'Intensità turistica (presenze)', 'Densità ricettiva', 'Utilizzazione lorda %'];
  const vals = [r.dt, r.it, r.dr, r.ul];
  const qlabel = DATA.meta.quad_labels[r.quad], qicon = DATA.meta.quad_icons[r.quad], qdesc = DATA.meta.quad_desc[r.quad];
  return `<div class="popup"><h3>${{c.n}}</h3><div class="sub">${{c.pv || ''}} · ${{qicon}} ${{qlabel}}</div>
  <div class="kv"><div><span>Superficie</span>${{fmt(c.area, 1)}} km²</div><div><span>Attrattività</span>${{fmt(c.attr, 0)}}/100</div></div>
  <div class="ind">${{labels.map((l, i) => `<div class="indrow"><span>${{l}}</span><div class="track"><div class="fill" style="width:${{(r.nz[i] || 0) * 100}}%"></div></div><b>${{fmt(vals[i], i === 1 ? 3 : 1)}}</b></div>`).join('')}}</div>
  <div class="bigscore"><span>Overtourism · Gem score</span><strong>${{fmt(r.ovt, 1)}}% · ${{fmt(r.gem, 1)}}</strong></div>
  <p class="whyp"><b>${{qicon}} ${{qlabel}}:</b> ${{qdesc}}.</p></div>`;
}}

// ---- Filtro per quadrante ("tutti i comuni" = tutti i chip attivi) ----
function buildQFilter() {{
  const chips = QUAD_KEYS.map(q => `<div class="qchip active" data-q="${{q}}" style="--qc:${{DATA.meta.quad_colors[q]}}">${{DATA.meta.quad_icons[q]}} ${{DATA.meta.quad_labels[q]}}</div>`).join('');
  $('qfilter').innerHTML = chips + `<span class="qfilter-sep">·</span><span class="qfilter-all" id="qAll">Tutti</span><span class="qfilter-sep">/</span><span class="qfilter-all" id="qNone">Nessuno</span>`;
  syncChips();
  $('qfilter').querySelectorAll('.qchip').forEach(chip => {{
    chip.onclick = () => {{
      const q = chip.dataset.q;
      if (selectedQuads.has(q)) selectedQuads.delete(q); else selectedQuads.add(q);
      syncChips();
      render();
    }};
  }});
  $('qAll').onclick = () => {{ selectedQuads = new Set(QUAD_KEYS); syncChips(); render(); }};
  $('qNone').onclick = () => {{ selectedQuads = new Set(); syncChips(); render(); }};
}}
function syncChips() {{
  $('qfilter').querySelectorAll('.qchip').forEach(chip => chip.classList.toggle('active', selectedQuads.has(chip.dataset.q)));
}}

// ---- Selezione di un comune (da classifica o da ricerca): centra la mappa e apre il popup ----
function selectComune(code) {{
  const m = markers.get(code);
  if (!m) return;
  if (!map.hasLayer(m)) m.addTo(map);
  map.setView(m.getLatLng(), 10, {{animate: true}});
  const entry = currentByCode.get(code);
  if (entry && entry.r) {{ m.openPopup(); }}
  else {{ m.openTooltip(); setTimeout(() => m.closeTooltip(), 2500); }}
}}

// ---- Ricerca comune ----
const SEARCH_INDEX = DATA.comuni.map(c => ({{i: c.i, n: c.n, pv: c.pv || ''}})).sort((a, b) => a.n.localeCompare(b.n, 'it'));
function renderSearchResults(query) {{
  const box = $('searchResults');
  const q = query.trim().toLowerCase();
  if (!q) {{ box.innerHTML = ''; box.classList.add('empty'); return; }}
  const matches = SEARCH_INDEX.filter(c => c.n.toLowerCase().includes(q)).slice(0, 12);
  if (!matches.length) {{
    box.innerHTML = '<div class="none">Nessun comune trovato</div>';
    box.classList.remove('empty');
    return;
  }}
  box.innerHTML = matches.map(c => `<div data-code="${{c.i}}"><span>${{c.n}}</span>${{c.pv ? `<span class="prov">· ${{c.pv}}</span>` : ''}}</div>`).join('');
  box.classList.remove('empty');
  box.querySelectorAll('div[data-code]').forEach(el => el.onclick = () => {{
    const code = Number(el.dataset.code);
    selectComune(code);
    $('searchInput').value = SEARCH_INDEX.find(c => c.i === code).n;
    box.innerHTML = ''; box.classList.add('empty');
  }});
}}
function wireSearch() {{
  $('searchInput').addEventListener('input', e => renderSearchResults(e.target.value));
  $('searchInput').addEventListener('keydown', e => {{
    if (e.key === 'Enter') {{
      const first = $('searchResults').querySelector('div[data-code]');
      if (first) first.click();
    }} else if (e.key === 'Escape') {{
      $('searchResults').innerHTML = ''; $('searchResults').classList.add('empty'); $('searchInput').blur();
    }}
  }});
  document.addEventListener('click', e => {{
    if (!e.target.closest('#searchwrap')) {{ $('searchResults').innerHTML = ''; $('searchResults').classList.add('empty'); }}
  }});
}}

function render() {{
  const idx = STEP_INDEX[year + '-' + month];
  const all = DATA.comuni.map(c => ({{c, r: rowFor(c, idx)}}));
  currentByCode = new Map(all.map(x => [x.c.i, x]));

  const visible = all.filter(x => x.r && selectedQuads.has(x.r.quad)).sort((a, b) => b.r.gem - a.r.gem);
  const visibleSet = new Set(visible.map(x => x.c.i));

  $('listTitle').textContent = selectedQuads.size === 4 ? 'Tutti i comuni' : 'Comuni selezionati';
  $('count').textContent = `${{visible.length}} comuni`;
  $('rankings').innerHTML = visible.map(({{c, r}}, i) => `<article class="card" data-code="${{c.i}}">
     <div class="rank">${{i + 1}}</div>
     <div class="cardtop"><span class="place">${{c.n}}</span><span class="score">${{fmt(r.gem, 1)}}/100</span></div>
     <div class="why">${{c.pv || ''}} · ${{DATA.meta.quad_icons[r.quad]}} ${{DATA.meta.quad_labels[r.quad]}} · overtourism ${{fmt(r.ovt, 1)}}%</div>
     <div class="bars">${{r.nz.map(v => `<i class="bar" style="width:${{Math.max(3, (v || 0) * 100)}}%"></i>`).join('')}}</div>
     <div class="meta"><span>ATTR ${{fmt(c.attr, 0)}}</span><span>rank ${{i + 1}}/${{visible.length}}</span></div>
   </article>`).join('') || '<div class="empty">Nessun comune corrisponde ai filtri selezionati.</div>';

  $('rankings').querySelectorAll('.card').forEach(el => el.onclick = () => selectComune(Number(el.dataset.code)));

  all.forEach(({{c, r}}) => {{
    let m = markers.get(c.i);
    const quad = r ? r.quad : null;
    const col = quad ? DATA.meta.quad_colors[quad] : '#9aa7ab';
    const rad = r ? markerRadius(r.gem) : 4;
    if (!m) {{
      m = L.circleMarker([c.lat, c.lng], {{radius: rad, color: '#fff', weight: 1, fillColor: col, fillOpacity: .85}});
      m.bindTooltip('', {{direction: 'top', offset: [0, -4]}});
      m.bindPopup('', {{maxWidth: 320}});
      markers.set(c.i, m);
    }}
    m.setStyle({{radius: rad, fillColor: col, fillOpacity: r ? .85 : .25}});
    m.setTooltipContent(r ? `<b>${{c.n}}</b><br>${{DATA.meta.quad_icons[quad]}} ${{DATA.meta.quad_labels[quad]}} · ${{fmt(r.gem, 1)}}` : `<b>${{c.n}}</b><br>dati non disponibili`);
    if (r) m.setPopupContent(popup(c, r));
    const showOnMap = r && visibleSet.has(c.i);
    if (showOnMap) {{ if (!map.hasLayer(m)) m.addTo(map); }} else {{ map.removeLayer(m); }}
  }});
}}

function buildLegend() {{
  const rows = ['1', '2', '3', '4'].map(q => `<div><i style="background:${{DATA.meta.quad_colors[q]}}"></i>${{DATA.meta.quad_icons[q]}} ${{DATA.meta.quad_labels[q]}}</div>`).join('');
  $('legend').innerHTML = `<b>Quadranti</b>${{rows}}`;
}}

function init() {{
  MONTHS.forEach((m, i) => $('month').add(new Option(m, i + 1)));
  YEARS.forEach(y => $('year').add(new Option(y, y)));
  $('month').value = month;
  $('year').value = year;
  $('month').onchange = e => {{ month = +e.target.value; render(); }};
  $('year').onchange = e => {{ year = +e.target.value; render(); }};

  map = L.map('map', {{zoomControl: false, center: [40, 9], zoom: 8, minZoom: 7}});
  L.control.zoom({{position: 'bottomright'}}).addTo(map);
  L.tileLayer('https://{{s}}.tile.openstreetmap.org/{{z}}/{{x}}/{{y}}.png', {{attribution: '© OpenStreetMap contributors', maxZoom: 19}}).addTo(map);
  regionLayer = L.geoJSON(DATA.region, {{style: {{color: '#1a5276', weight: 2, opacity: .8, fillColor: '#48c9b0', fillOpacity: .06}}}}).addTo(map);
  map.fitBounds(regionLayer.getBounds(), {{padding: [25, 25]}});

  buildQFilter();
  wireSearch();
  buildLegend();
  render();
  setTimeout(() => {{ $('loading').style.display = 'none'; map.invalidateSize(); }}, 250);
}}

init();
</script>
</body>
</html>
"""


html = build_html(data_json)
with open(OUT_HTML, "w", encoding="utf-8") as f:
    f.write(html)

print(f"HTML scritto in {OUT_HTML.resolve()}  ({os.path.getsize(OUT_HTML)/1024:.1f} KB)")

HTML scritto in /home/davide/codice/progetto_git/sardegna-overtourism-aida26/data/mappa_gemme_nascoste/mappa_gemme_nascoste.html  (1288.6 KB)


## 8 · Validazione

Verifica finale: il JSON incorporato nell'HTML è valido e riporta il numero atteso di comuni/step;
il file è pronto per essere aperto in un browser.

In [22]:
import re

html_check = OUT_HTML.read_text(encoding='utf-8')
m = re.search(r'<script type="application/json" id="embedded-data">(.*?)</script>', html_check, re.S)
parsed = json.loads(m.group(1))

assert len(parsed['comuni']) == len(comuni), "numero di comuni incoerente"
assert len(parsed['meta']['steps']) == n_steps, "numero di step incoerente"
assert parsed['region']['type'] in ('Polygon', 'MultiPolygon')

print("✅ JSON incorporato valido")
print(f"   Comuni  : {len(parsed['comuni'])}")
print(f"   Step    : {len(parsed['meta']['steps'])}")
print(f"   File    : {OUT_HTML.resolve()}  ({OUT_HTML.stat().st_size/1024:.1f} KB)")
print("\nApri il file in un browser per usare la mappa (selettori mese/anno, classifica gemme, popup).")

✅ JSON incorporato valido
   Comuni  : 371
   Step    : 48
   File    : /home/davide/codice/progetto_git/sardegna-overtourism-aida26/data/mappa_gemme_nascoste/mappa_gemme_nascoste.html  (1288.6 KB)

Apri il file in un browser per usare la mappa (selettori mese/anno, classifica gemme, popup).
